# Ship Detection in SAR Imagery Using Deep Learning
**Faster R-CNN | ResNet-50 + FPN | SSDD Dataset**

---
## Setup: Upload your project folder to Colab

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Copy project from Drive to Colab
# OPTION A: If you uploaded SARDATA folder to Google Drive root
!cp -r /content/drive/MyDrive/SARDATA /content/SARDATA

# OPTION B: If you zipped it as SARDATA.zip and uploaded to Drive
# !unzip /content/drive/MyDrive/SARDATA.zip -d /content/

%cd /content/SARDATA

In [ ]:
# Step 3: Install dependencies
!pip install -q torch torchvision albumentations opencv-python tqdm matplotlib fpdf2

In [ ]:
# Step 4: Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---
## Phase 1: Dataset Verification

In [ ]:
# Verify dataset
import os
import json

data_root = 'data/sar-ship-detection-dataset'
ssdd_dir = os.path.join(data_root, 'SSDD')

train_imgs = len(os.listdir(os.path.join(ssdd_dir, 'images', 'train')))
test_imgs = len(os.listdir(os.path.join(ssdd_dir, 'images', 'test')))

with open(os.path.join(ssdd_dir, 'annotations', 'train.json')) as f:
    train_ann = json.load(f)
with open(os.path.join(ssdd_dir, 'annotations', 'test.json')) as f:
    test_ann = json.load(f)

print(f'Train: {train_imgs} images, {len(train_ann["annotations"])} annotations')
print(f'Test:  {test_imgs} images, {len(test_ann["annotations"])} annotations')
print(f'Categories: {train_ann["categories"]}')

In [ ]:
# Visualize sample images
import cv2
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
img_dir = os.path.join(ssdd_dir, 'images', 'train')
img_files = sorted(os.listdir(img_dir))[:4]

for ax, fname in zip(axes, img_files):
    img = cv2.imread(os.path.join(img_dir, fname), cv2.IMREAD_UNCHANGED)
    ax.imshow(img, cmap='gray')
    ax.set_title(fname, fontsize=10)
    ax.axis('off')

plt.suptitle('SSDD Training Samples', fontsize=14)
plt.tight_layout()
plt.show()

---
## Phase 2: Model Training (GPU)

In [ ]:
# Quick test - verify everything works (1 epoch)
!python main.py --mode train --epochs 1 --batch-size 4

In [ ]:
# FULL TRAINING - 50 epochs on GPU
# This takes ~30-45 min on Colab T4 GPU
!python main.py --mode train --epochs 50 --batch-size 4 --lr 0.005

In [ ]:
# View training curves
from IPython.display import Image, display
if os.path.exists('results/training_curves.png'):
    display(Image('results/training_curves.png'))
else:
    print('Training curves not found. Run training first.')

In [ ]:
# View training history
if os.path.exists('results/training_history.json'):
    with open('results/training_history.json') as f:
        history = json.load(f)
    print(f'Best mAP@0.5: {max(history["val_map"]):.4f}')
    print(f'Best Epoch: {history["val_map"].index(max(history["val_map"])) + 1}')
    print(f'Final Loss: {history["train_loss"][-1]:.4f}')
    print(f'Final Precision: {history["val_precision"][-1]:.4f}')
    print(f'Final Recall: {history["val_recall"][-1]:.4f}')

---
## Phase 3: Evaluation

In [ ]:
# Evaluate on test set
!python main.py --mode eval --model-path checkpoints/best_model.pth --split test

In [ ]:
# Evaluate on test_inshore and test_offshore subsets
!python main.py --mode eval --model-path checkpoints/best_model.pth --split test_inshore
print('---')
!python main.py --mode eval --model-path checkpoints/best_model.pth --split test_offshore

---
## Phase 4: Visualization

In [ ]:
# Generate visualizations
!python main.py --mode visualize --model-path checkpoints/best_model.pth --output-dir results/visualizations

In [ ]:
# Display visualizations
vis_dir = 'results/visualizations'
if os.path.exists(vis_dir):
    vis_files = [f for f in os.listdir(vis_dir) if f.endswith('.png')]
    fig, axes = plt.subplots(1, min(len(vis_files), 5), figsize=(25, 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    for ax, fname in zip(axes, vis_files[:5]):
        img = plt.imread(os.path.join(vis_dir, fname))
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle('Detection Results (Red=Predicted, Green=Ground Truth)', fontsize=14)
    plt.tight_layout()
    plt.show()

---
## Phase 5: Generate Technical Report (PDF)

In [ ]:
!python scripts/generate_report.py

In [ ]:
# Download report
from google.colab import files
files.download('results/SAR_Ship_Detection_Report.pdf')

---
## Phase 6: Save Everything Back to Drive

In [ ]:
# Copy trained model + results back to Google Drive
!cp -r /content/SARDATA/checkpoints /content/drive/MyDrive/SARDATA/checkpoints
!cp -r /content/SARDATA/results /content/drive/MyDrive/SARDATA/results
print('Saved checkpoints and results to Google Drive!')

In [ ]:
# Download model directly
from google.colab import files
if os.path.exists('checkpoints/best_model.pth'):
    files.download('checkpoints/best_model.pth')
    print('Downloaded best_model.pth')